In [1]:
# !pip install ccxt

In [2]:
import ccxt
import pandas as pd
import time

exchange = ccxt.binance()

symbol = 'BTC/USDT'
timeframe = '30m'

In [3]:


since = exchange.parse8601('2021-01-01T00:00:00Z')
all_data = []

while True:
    data = exchange.fetch_ohlcv(symbol, timeframe, since, limit=1000)
    if not data:
        break

    all_data += data
    since = data[-1][0] + 1
    time.sleep(0.2)

df = pd.DataFrame(all_data, columns=['timestamp','open','high','low','close','volume'])
df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')

In [4]:
df

,timestamp,open,high,low,close,volume,datetime
0,1609459200000,28923.63,29017.50,28690.17,28836.63,1320.688748,2021-01-01 00:00:00
1,1609461000000,28836.63,29031.34,28836.62,28995.13,991.122697,2021-01-01 00:30:00
2,1609462800000,28995.13,29417.87,28960.35,29385.39,3449.118444,2021-01-01 01:00:00
3,1609464600000,29387.07,29470.00,29311.75,29409.99,1953.950027,2021-01-01 01:30:00
4,1609466400000,29410.00,29465.26,29201.78,29298.86,1244.624776,2021-01-01 02:00:00
...,...,...,...,...,...,...,...
92994,1776907800000,78087.43,78430.23,78066.41,78359.23,276.071130,2026-04-23 01:30:00
92995,1776909600000,78359.71,78359.72,77987.42,78036.00,252.573330,2026-04-23 02:00:00
92996,1776911400000,78035.99,78112.35,77761.20,77939.92,255.749940,2026-04-23 02:30:00
92997,1776913200000,77939.91,78025.99,77623.34,77687.86,291.782910,2026-04-23 03:00:00


In [5]:
def calculate_kama(price, length, fast, slow):
    change = abs(price - price.shift(length))
    volatility = abs(price - price.shift(1)).rolling(length).sum()

    er = change / volatility
    er = er.replace([np.inf, -np.inf], 0).fillna(0)

    fast_sc = 2 / (fast + 1)
    slow_sc = 2 / (slow + 1)

    sc = (er * (fast_sc - slow_sc) + slow_sc) ** 2

    kama = pd.Series(index=price.index, dtype='float64')
    kama.iloc[length] = price.iloc[length]  # initial value

    for i in range(length + 1, len(price)):
        kama.iloc[i] = kama.iloc[i-1] + sc.iloc[i] * (price.iloc[i] - kama.iloc[i-1])

    return kama

In [24]:
import numpy as np
kama_fast = calculate_kama(df['close'], length=7, fast=2, slow=10)
kama_slow = calculate_kama(df['close'], length=21, fast=4, slow=10)

# Direct signal column
df['signal'] = 0

# Buy = 1
df.loc[
    (kama_fast > kama_slow) &
    (kama_fast.shift(1) <= kama_slow.shift(1)),
    'signal'] = 1

# Sell = -1
df.loc[
    (kama_fast < kama_slow) &
    (kama_fast.shift(1) >= kama_slow.shift(1)),
    'signal'] = -1

df['signal'] = df['signal'].replace(0, np.nan).ffill().fillna(0)
df = df[df['signal'] != 0].reset_index(drop=True)

df_filtered = df[df['signal'] != df['signal'].shift()].reset_index(drop=True)
df_filtered = df[df['signal'] != df['signal'].shift()].reset_index(drop=True)
df_filtered['next_price'] = df_filtered['close'].shift(-1)
df_filtered['points'] = (df_filtered['next_price'] - df_filtered['close']) * df_filtered['signal']

In [25]:
df_filtered['points'].sum()

np.float64(-11762.2300000002)